In [1]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import fsspec
import os
warnings.simplefilter('ignore') # filter some warning messages
xr.set_options(display_style="html")  #display dataset nicely 

In [2]:
%%time

import coiled

cluster_type = "Coiled"

if cluster_type == "Coiled":
    cluster = coiled.Cluster(
        region="eu-central-1",
        arm=True,  # run on ARM to save energy & cost
        worker_vm_types=["t4g.medium"],  # cheap ARM instances
        worker_options={"nthreads": 2},
        n_workers=200,
        name='Oriana',
        wait_for_workers=False,
        compute_purchase_option="spot_with_fallback",
        software="protocoast-notebook-arm",
        workspace="esip-lab",
        timeout=180  # keep cluster alive for 3 min
    )

Output()

CPU times: user 12.2 s, sys: 1.04 s, total: 13.2 s
Wall time: 2min 10s


In [3]:
client = cluster.get_client()
client

<Client: 'tls://10.0.47.152:8786' processes=109 threads=218, memory=359.37 GiB>

In [4]:
%%time

ds_sst = xr.open_zarr('https://mur-sst.s3.us-west-2.amazonaws.com/zarr-v1',consolidated=True)

ds_sst

CPU times: user 5.26 s, sys: 834 ms, total: 6.09 s
Wall time: 33.3 s


<xarray.Dataset> Size: 117TB
Dimensions:           (time: 6443, lat: 17999, lon: 36000)
Coordinates:
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
  * time              (time) datetime64[ns] 52kB 2002-06-01T09:00:00 ... 2020...
Data variables:
    analysed_sst      (time, lat, lon) float64 33TB dask.array<chunksize=(5, 1799, 3600), meta=np.ndarray>
    analysis_error    (time, lat, lon) float64 33TB dask.array<chunksize=(5, 1799, 3600), meta=np.ndarray>
    mask              (time, lat, lon) float32 17TB dask.array<chunksize=(5, 1799, 3600), meta=np.ndarray>
    sea_ice_fraction  (time, lat, lon) float64 33TB dask.array<chunksize=(5, 1799, 3600), meta=np.ndarray>
Attributes: (12/47)
    Conventions:                CF-1.7
    Metadata_Conventions:       Unidata Observation Dataset v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    MUR = "Multi-scale Ultra-high Resolution"
    creator_email:              ghrsst@podaac.jpl.nasa.gov
    ...                         ...
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    time_coverage_end:          20200116T210000Z
    time_coverage_start:        20200115T210000Z
    title:                      Daily MUR SST, Final product
    uuid:                       27665bc0-d5fc-11e1-9b23-0800200c9a66
    westernmost_longitude:      -180.0

In [5]:
sst = ds_sst['analysed_sst']
sst_subset = sst.sel(
    lat=slice(-20, 20),   # reversed if lat is descending
    lon=slice(100, 160)
)

In [6]:
# MONTHLY MEAN 
sst_monthly = sst_subset.resample(time='1MS').mean('time', keep_attrs=True, skipna=False)

# MONTHLY CLIMATOLOGY ---
climatology_mean_monthly = sst_monthly.groupby('time.month').mean('time', keep_attrs=True, skipna=False)

# MONTHLY ANOMALY ---
sst_anomaly_monthly = (sst_monthly.groupby('time.month') - climatology_mean_monthly)

# OUTPUT 
sst_anomaly_monthly

<xarray.DataArray 'analysed_sst' (time: 212, lat: 4001, lon: 6001)> Size: 41GB
dask.array<sub, shape=(212, 4001, 6001), dtype=float64, chunksize=(1, 1799, 3600), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 16kB -20.0 -19.99 -19.98 -19.97 ... 19.98 19.99 20.0
  * lon      (lon) float32 24kB 100.0 100.0 100.0 100.0 ... 160.0 160.0 160.0
  * time     (time) datetime64[ns] 2kB 2002-06-01 2002-07-01 ... 2020-01-01
    month    (time) int64 2kB 6 7 8 9 10 11 12 1 2 3 ... 4 5 6 7 8 9 10 11 12 1

In [ ]:
import hvplot.xarray
import holoviews as hv

hv.extension('bokeh')

# --- Select ONLY 2015 and reduce time ---
d_spatial = sst_anomaly_monthly.sel(time="2015").mean(dim="time")

# --- HVPlot geospatial map ---
d_spatial.hvplot.image(
    x='lon',
    y='lat',
    geo=True,
    tiles='OSM',
    rasterize=True,
    cmap='RdBu_r',
    alpha=0.8,
    frame_width=500
)